# Labelled Clip Anomaly Classification — frozen ResNet50 + supervised classifier

This is an adapted copy of `demo_notebooks/AnomalyClassification/01_ImageAnomalyClassification.ipynb`.
It keeps the demo's frozen ResNet50 feature extraction, but uses an ordinary supervised classifier
trained from the labelled videos under `stoppage_detection_and_classification/cnn_classifier/data/training_clips`.

Each video is represented by the mean frozen-ResNet feature of six frames selected with the same
NCC onset/peak selector used by the anomaly-classification video demos. Labels come only from the
three folder names. No mask, contour, `minAreaRect`, or hand-written tilt threshold is used.

Videos timestamped before **2026-07-01 00:00:00** are rotated 180° immediately after decoding.
This orientation correction therefore applies before both NCC selection and ResNet encoding.

There is no one-shot evaluation. Repeated stratified validation starts automatically once every
class has at least three clips. Until then, the notebook fits a provisional model on all labelled
clips but does not present its training accuracy as evidence of generalisation.


## Setup

Use the project `24h_insights` kernel. The first ResNet run downloads the standard torchvision
weights into `stoppage_detection_and_classification/cnn_classifier/output/model_cache`; later runs reuse the local cache.


In [ ]:
import os
import re
import sys
from collections import Counter
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Find the repository root whether Jupyter starts in the repo or a pipeline folder.
working_directory = Path.cwd().resolve()
repository_candidates = [working_directory, *working_directory.parents]
REPO_ROOT = next(
    candidate
    for candidate in repository_candidates
    if (candidate / "VideoModule").is_dir()
)
PIPELINE_ROOT = REPO_ROOT / "stoppage_detection_and_classification"
CNN_CLASSIFIER_DIR = PIPELINE_ROOT / "cnn_classifier"
DEMO_DIR = REPO_ROOT / "demo_notebooks" / "AnomalyClassification"
for import_path in (REPO_ROOT, CNN_CLASSIFIER_DIR, DEMO_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

import _shared as S

from VideoModule.anomaly_classification.image_fewshot import frozen_resnet_embedder
from VideoModule.anomaly_classification.supervised_eval import map_clusters_to_labels
from VideoModule.embeddings.frame_embeddings.changepoint_selector import (
    _NCC_DOWN,
    select_peak_pair_window,
)
from VideoModule.io.decode import decode_clip_rgb_with_fps
from VideoModule.plotting import render_score_bars
from VideoModule.plotting.confusion_matrix_plot import render_confusion_matrix
from VideoModule.plotting.event_selection_plots import render_paired_frame_grid
from VideoModule.statistical_analysis.clustering import make_clusterer

TEST_CLIPS_DIR = CNN_CLASSIFIER_DIR / "data" / "training_clips"
OUTPUT_DIR = CNN_CLASSIFIER_DIR / "output" / "04_train_fewshot_anomaly_classifier"
DEBUG_DIR = OUTPUT_DIR / "selected_frame_grids"
MODEL_CACHE_DIR = CNN_CLASSIFIER_DIR / "output" / "model_cache"
FEATURE_CACHE_PATH = OUTPUT_DIR / "clip_features.npz"
MODEL_PATH = OUTPUT_DIR / "supervised_clip_classifier.joblib"
for folder in (OUTPUT_DIR, DEBUG_DIR, MODEL_CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Keep downloaded model weights in the writable project workspace.
os.environ.setdefault("TORCH_HOME", str(MODEL_CACHE_DIR / "torch"))

N_SELECT = 120
PCA_COMPONENTS = 10
MIN_GROUPS_PER_CLASS_FOR_CV = 3
MIN_CLIPS_PER_CLASS_FOR_CV = MIN_GROUPS_PER_CLASS_FOR_CV
CV_REPEATS = 5
OVERLAP_TOLERANCE_SECONDS = 0.0
C = 1
# PCA_COMPONENTS = [5, 10, 20, 40]
# C = [0.01, 0.1, 1, 10]

# Correct the older inverted-camera recordings before any image analysis.
ROTATE_BEFORE_TIMESTAMP = pd.Timestamp("2026-07-01 00:00:00")
VIDEO_TIMESTAMP_PATTERN = re.compile(
    r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_\d{6})"
)

CLASS_FOLDERS = {
    "normal": 0,
    "fallen_before_entry": 1,
    "fallen_in_view": 2,
}
CLASS_NAMES = {
    0: "normal",
    1: "fallen_on_entry",
    2: "fell_during_view",
}
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".ts", ".m4v"}


def parse_video_timestamp(video_path):
    # Use the recording timestamp embedded in the filename, not its modification time.
    timestamp_match = VIDEO_TIMESTAMP_PATTERN.search(Path(video_path).name)
    if timestamp_match is None:
        raise ValueError(f"No recording timestamp found in video filename: {video_path}")
    return pd.to_datetime(
        timestamp_match.group(1),
        format="%Y-%m-%d_%H-%M-%S_%f",
    )


def read_video_duration_seconds(video_path):
    # Read only container metadata; decoding every frame is unnecessary for duration.
    video_capture = cv2.VideoCapture(str(video_path))
    if not video_capture.isOpened():
        raise ValueError(f"Could not open video while reading duration: {video_path}")

    try:
        frame_count = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = float(video_capture.get(cv2.CAP_PROP_FPS))
    finally:
        video_capture.release()

    if frame_count <= 0 or fps <= 0:
        raise ValueError(f"Invalid frame count or FPS while reading duration: {video_path}")
    return frame_count / fps


def format_duration(total_seconds):
    # Format accumulated footage length as hours:minutes:seconds.
    rounded_seconds = int(round(total_seconds))
    hours, remaining_seconds = divmod(rounded_seconds, 3600)
    minutes, seconds = divmod(remaining_seconds, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


if not TEST_CLIPS_DIR.is_dir():
    raise FileNotFoundError(f"Labelled test-clips folder does not exist: {TEST_CLIPS_DIR}")

S.print_hardware_banner()
print(f"Labelled clips: {TEST_CLIPS_DIR}")
print(f"Outputs:        {OUTPUT_DIR}")
print(f"Rotate 180° before: {ROTATE_BEFORE_TIMESTAMP}")


## Dataset inventory

Summarise the labelled footage before running NCC or ResNet. Duration comes from each video's frame
count and FPS. The table reports clip count, clips requiring 180° correction, and total footage for
each class.


In [ ]:
def recording_source_id(video_path):
    # Use the camera identifier so only clips from the same source can overlap.
    camera_match = re.search(r'(cortexvpu-[^_]+)', Path(video_path).name)
    if camera_match is not None:
        return camera_match.group(1)
    return Path(video_path).stem


def assign_recording_groups(records, overlap_tolerance_seconds=0.0):
    # Assign one group to each transitive chain of overlapping clips.
    indices_by_source = {}
    for record_index, record in enumerate(records):
        source_id = recording_source_id(record['video_path'])
        indices_by_source.setdefault(source_id, []).append(record_index)

    group_by_index = {}
    overlap_tolerance = pd.Timedelta(seconds=overlap_tolerance_seconds)

    for source_id, source_indices in indices_by_source.items():
        sorted_indices = sorted(
            source_indices,
            key=lambda index: records[index]['video_timestamp'],
        )
        group_number = 0
        active_group_end = None

        for record_index in sorted_indices:
            record = records[record_index]
            clip_start = record['video_timestamp']
            clip_end = clip_start + pd.Timedelta(
                seconds=record['duration_seconds'],
            )

            starts_new_group = (
                active_group_end is None
                or clip_start >= active_group_end + overlap_tolerance
            )
            if starts_new_group:
                group_number += 1
                active_group_end = clip_end
            else:
                active_group_end = max(active_group_end, clip_end)

            group_by_index[record_index] = (
                f'{source_id}:overlap-{group_number:04d}'
            )

    return [group_by_index[index] for index in range(len(records))]


clip_inventory = []
for folder_name, class_code in CLASS_FOLDERS.items():
    class_folder = TEST_CLIPS_DIR / folder_name
    if not class_folder.is_dir():
        raise FileNotFoundError(f"Missing labelled class folder: {class_folder}")

    # Record the metadata needed by both the summary and downstream training cells.
    for video_path in sorted(class_folder.rglob("*")):
        if not video_path.is_file() or video_path.suffix.lower() not in VIDEO_EXTENSIONS:
            continue

        video_timestamp = parse_video_timestamp(video_path)
        clip_inventory.append({
            "folder_name": folder_name,
            "class_code": class_code,
            "class_name": CLASS_NAMES[class_code],
            "video_path": video_path,
            "video_timestamp": video_timestamp,
            "rotate_180": video_timestamp < ROTATE_BEFORE_TIMESTAMP,
            "duration_seconds": read_video_duration_seconds(video_path),
            "modified_ns": video_path.stat().st_mtime_ns,
        })

if not clip_inventory:
    raise ValueError(f"No labelled video clips found under {TEST_CLIPS_DIR}")

# Keep every labelled clip, including overlapping normal clips.
# Overlapping clips share a group so they stay on the same side of each CV fold.
recording_group_ids = assign_recording_groups(
    clip_inventory,
    overlap_tolerance_seconds=OVERLAP_TOLERANCE_SECONDS,
)
for record, recording_group_id in zip(clip_inventory, recording_group_ids):
    record['recording_group'] = recording_group_id

print(f'All clips retained:               {len(clip_inventory)}')
print(f'Independent recording groups:     {len(set(recording_group_ids))}')

# Preserve the configured class order instead of sorting labels alphabetically.
summary_rows = []
for folder_name, class_code in CLASS_FOLDERS.items():
    class_records = [
        record
        for record in clip_inventory
        if record["class_code"] == class_code
    ]
    class_duration_seconds = sum(
        record["duration_seconds"]
        for record in class_records
    )
    summary_rows.append({
        "class": CLASS_NAMES[class_code],
        "total_clips": len(class_records),
        "clips_rotated_180": sum(record["rotate_180"] for record in class_records),
        "footage_seconds": round(class_duration_seconds, 1),
        "total_footage_hh_mm_ss": format_duration(class_duration_seconds),
    })

dataset_summary = pd.DataFrame(summary_rows)
total_clips = len(clip_inventory)
total_rotated_clips = sum(record["rotate_180"] for record in clip_inventory)
total_footage_seconds = sum(record["duration_seconds"] for record in clip_inventory)

print(f"Total clips:            {total_clips}")
print(f"Total clips rotated:    {total_rotated_clips}")
print(f"Total footage:          {format_duration(total_footage_seconds)}")
display(dataset_summary)

## 1. The data — labelled video clips

Folder names provide the labels. This replaces the synthetic still-image manifest in the demo.
The timestamp embedded in each filename controls the date-based 180° orientation correction. The
filename, modification time, and orientation rule together form the feature-cache fingerprint.


In [ ]:
clip_records = []
for inventory_record in clip_inventory:
    folder_name = inventory_record["folder_name"]
    video_path = inventory_record["video_path"]
    class_code = inventory_record["class_code"]
    clip_records.append({
        "clip_key": f"{folder_name}/{video_path.name}",
        "clip_id": f"{folder_name}__{video_path.stem}",
        "video_path": video_path,
        "video_timestamp": inventory_record["video_timestamp"],
        "rotate_180": inventory_record["rotate_180"],
        "duration_seconds": inventory_record["duration_seconds"],
        "class_code": class_code,
        "class_name": CLASS_NAMES[class_code],
        "modified_ns": inventory_record["modified_ns"],
    })

clip_ids = [record["clip_id"] for record in clip_records]
recording_groups = np.array([record['recording_group'] for record in clip_inventory])
if len(clip_ids) != len(set(clip_ids)):
    raise ValueError("Clip IDs are not unique; rename duplicate files before training")

labels = np.array([record["class_code"] for record in clip_records], dtype=np.int64)
classes = sorted(set(labels.tolist()))
class_counts = Counter(labels.tolist())
by_class = {class_code: np.where(labels == class_code)[0] for class_code in classes}

clip_table = pd.DataFrame([
    {
        "class": record["class_name"],
        "video": record["video_path"].name,
        "timestamp": record["video_timestamp"],
        "rotated_180": record["rotate_180"],
        "duration_seconds": round(record["duration_seconds"], 2),
        "path": record["video_path"],
    }
    for record in clip_records
])
display(clip_table)
print("Class counts:", {CLASS_NAMES[code]: class_counts[code] for code in classes})
print(f"Clips rotated 180°: {sum(record['rotate_180'] for record in clip_records)}")


## 2. Correct orientation and select each clip's event frames with NCC

Clips recorded before the cutoff are rotated 180° as complete frames immediately after decoding.
The NCC selector therefore always sees orientation-corrected footage and chooses six onset/peak
frames from that corrected view. The saved diagnostic grids use the same orientation.


In [ ]:
def apply_video_orientation(frames, rotate_180):
    # Apply one consistent transform to every frame so no artificial motion is introduced.
    if not rotate_180:
        return frames
    return [cv2.rotate(frame, cv2.ROTATE_180) for frame in frames]


def select_clip_frames(video_path, clip_id, rotate_180):
    # Decode and orient the clip before calculating the NCC onset/peak selection.
    frames, fps = decode_clip_rgb_with_fps(video_path)
    frames = apply_video_orientation(frames, rotate_180)
    small_frames = [cv2.resize(frame, (_NCC_DOWN, _NCC_DOWN)) for frame in frames]
    selection = select_peak_pair_window(small_frames, fps, N_SELECT)
    if not selection.window:
        raise ValueError(f"NCC selected no frames for {video_path}")

    # Save the same role-labelled grid used by the demo notebooks.
    grid_path = DEBUG_DIR / f"{clip_id}_grid.png"
    render_paired_frame_grid(frames, selection.pairs, clip_id, grid_path)
    selected_frames = [frames[index] for index in selection.window]
    return selected_frames, selection, grid_path


selected_frames_by_clip = {}
selection_rows = []
for record in clip_records:
    selected_frames, selection, grid_path = select_clip_frames(
        record["video_path"],
        record["clip_id"],
        record["rotate_180"],
    )
    selected_frames_by_clip[record["clip_id"]] = selected_frames
    selection_rows.append({
        "class": record["class_name"],
        "video": record["video_path"].name,
        "rotated_180": record["rotate_180"],
        "selected_indices": selection.window,
        "grid_path": grid_path,
    })

selection_df = pd.DataFrame(selection_rows)
display(selection_df)
for grid_path in selection_df["grid_path"].head(3):
    display(Image(filename=str(grid_path)))


## 3. Frozen ResNet50 clip features

This preserves the demo's recommended safe default: no gradient updates. Each orientation-corrected
selected frame is embedded independently, then the six vectors are averaged and L2-normalised into
one clip vector. The cache fingerprint includes the date cutoff and each clip's rotation decision,
so an embedding made from the old orientation cannot be reused.


In [ ]:
def clip_cache_fingerprint(records):
    orientation_rule = f"rotate-before={ROTATE_BEFORE_TIMESTAMP.isoformat()}"
    orientation_rule += f'|n-select={N_SELECT}'
    return np.array([
        (
            f"{record['clip_key']}|{record['modified_ns']}|{orientation_rule}|"
            f"rotate-180={int(record['rotate_180'])}"
        )
        for record in records
    ])


def cache_matches(records):
    if not FEATURE_CACHE_PATH.exists():
        return False
    cached = np.load(FEATURE_CACHE_PATH, allow_pickle=True)
    return np.array_equal(cached["fingerprint"], clip_cache_fingerprint(records))


_resnet_embedder = None


def get_resnet_embedder():
    global _resnet_embedder
    if _resnet_embedder is None:
        device = "cuda" if S.cuda_available() else "cpu"
        _resnet_embedder = frozen_resnet_embedder(device=device)
    return _resnet_embedder


if cache_matches(clip_records):
    cached = np.load(FEATURE_CACHE_PATH, allow_pickle=True)
    CLIP_FEATURES = cached["features"].astype(np.float32)
    print(f"Loaded cached clip features: {FEATURE_CACHE_PATH}")
else:
    # Flatten selected frames so ResNet processes them in one efficient batch sequence.
    all_selected_frames = []
    clip_slices = {}
    for record in clip_records:
        clip_frames = selected_frames_by_clip[record["clip_id"]]
        slice_start = len(all_selected_frames)
        all_selected_frames.extend(clip_frames)
        clip_slices[record["clip_id"]] = slice(slice_start, len(all_selected_frames))

    with S.timed(f"ResNet50 encode ({len(all_selected_frames)} selected frames)"):
        frame_features = get_resnet_embedder()(all_selected_frames)

    # Pool each clip's frame features and normalise the final clip vector.
    pooled_features = []
    for record in clip_records:
        clip_feature = frame_features[clip_slices[record["clip_id"]]].mean(axis=0)
        feature_norm = np.linalg.norm(clip_feature)
        if feature_norm > 0:
            clip_feature = clip_feature / feature_norm
        pooled_features.append(clip_feature.astype(np.float32))
    CLIP_FEATURES = np.stack(pooled_features)

    np.savez_compressed(
        FEATURE_CACHE_PATH,
        features=CLIP_FEATURES,
        fingerprint=clip_cache_fingerprint(clip_records),
        clip_ids=np.array(clip_ids),
        labels=labels,
    )
    print(f"Saved clip feature cache: {FEATURE_CACHE_PATH}")

print(f"Clip feature matrix: {CLIP_FEATURES.shape}")


## 4. Zero-shot baseline — cluster without labels

This is the demo's k=0 baseline. Labels are used only after clustering to map each cluster to its
majority class and calculate a confusion matrix.


In [ ]:
with S.timed("zero-shot clustering"):
    cluster_result = make_clusterer(
        "agglomerative_k",
        n_clusters=len(classes),
    )(CLIP_FEATURES)

cluster_labels = np.asarray(cluster_result.labels)
cluster_to_class = map_clusters_to_labels(cluster_labels, labels.tolist())
zero_shot_predictions = np.array([
    cluster_to_class[cluster_id]
    for cluster_id in cluster_labels
])
zero_shot_f1 = f1_score(
    labels,
    zero_shot_predictions,
    labels=classes,
    average="macro",
    zero_division=0,
)
zero_shot_confusion = confusion_matrix(labels, zero_shot_predictions, labels=classes)
print(f"Zero-shot macro-F1: {zero_shot_f1:.3f}")

## 5. Supervised classification — no one-shot sampling

The classifier uses all available training clips in each fold. Once every class has at least three
clips, the notebook runs repeated stratified cross-validation. PCA is fitted inside each fold so
information from its test clips cannot leak into training.

With fewer than three clips in any class, evaluation is skipped. A model can still be fitted on all
labelled data for experimentation, but its training-set result is only a sanity check.


In [ ]:
def make_supervised_classifier(training_sample_count):
    # Keep PCA small enough to fit every training fold safely.
    component_count = min(
        PCA_COMPONENTS,
        training_sample_count - 1,
        CLIP_FEATURES.shape[1],
    )
    if component_count < 1:
        raise ValueError("At least two training clips are required")

    # Fit every preprocessing step inside the pipeline to prevent test leakage.
    return make_pipeline(
        PCA(n_components=component_count, random_state=0),
        StandardScaler(),
        LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=2000,
            random_state=0,
        ),
    )


class_group_counts = {
    class_code: len(set(recording_groups[labels == class_code]))
    for class_code in classes
}
smallest_class_group_count = min(class_group_counts.values())
smallest_class_count = smallest_class_group_count
supervised_evaluation_available = (
    smallest_class_group_count >= MIN_GROUPS_PER_CLASS_FOR_CV
)

if supervised_evaluation_available:
    # Keep every overlapping recording group entirely on one side of each fold.
    split_count = min(5, smallest_class_group_count)
    grouped_splits = []
    for repeat_index in range(CV_REPEATS):
        repeat_splitter = StratifiedGroupKFold(
            n_splits=split_count,
            shuffle=True,
            random_state=repeat_index,
        )
        grouped_splits.extend(repeat_splitter.split(
            CLIP_FEATURES,
            labels,
            groups=recording_groups,
        ))

    supervised_scores = []
    supervised_confusion = np.zeros((len(classes), len(classes)), dtype=int)
    for train_indices, test_indices in grouped_splits:
        training_groups = set(recording_groups[train_indices])
        test_groups = set(recording_groups[test_indices])
        if not training_groups.isdisjoint(test_groups):
            raise AssertionError('A recording group crossed the validation boundary')
        fold_classifier = make_supervised_classifier(len(train_indices))
        fold_classifier.fit(CLIP_FEATURES[train_indices], labels[train_indices])
        fold_predictions = fold_classifier.predict(CLIP_FEATURES[test_indices])

        supervised_scores.append(f1_score(
            labels[test_indices],
            fold_predictions,
            labels=classes,
            average="macro",
            zero_division=0,
        ))
        supervised_confusion += confusion_matrix(
            labels[test_indices],
            fold_predictions,
            labels=classes,
        )

    supervised_mean = float(np.mean(supervised_scores))
    supervised_std = float(np.std(supervised_scores))
    print(
        f"Repeated {split_count}-fold supervised macro-F1: "
        f"{supervised_mean:.3f} +/- {supervised_std:.3f}"
    )
else:
    supervised_mean = None
    supervised_std = None
    supervised_confusion = None
    print(
        "Supervised validation skipped: each class needs at least "
        f"{MIN_CLIPS_PER_CLASS_FOR_CV} clips. "
        f"The smallest class currently has {smallest_class_count}."
    )


## 6. Review the evaluation

The clustering result remains a descriptive baseline. The supervised confusion matrix and score
comparison are produced only when repeated stratified validation is possible; no one-shot result
is calculated or displayed.


In [ ]:
class_axis_labels = [CLASS_NAMES[code] for code in classes]

# Always save the descriptive clustering baseline.
zero_shot_path = OUTPUT_DIR / "confusion_zero_shot.png"
render_confusion_matrix(
    zero_shot_confusion,
    class_axis_labels,
    zero_shot_path,
    title=f"zero-shot clustering (macro-F1={zero_shot_f1:.2f})",
)
display(Image(filename=str(zero_shot_path)))

if supervised_evaluation_available:
    # Save supervised results only when every class has enough independent clips.
    supervised_path = OUTPUT_DIR / "confusion_supervised_cv.png"
    render_confusion_matrix(
        supervised_confusion,
        class_axis_labels,
        supervised_path,
        title=f"supervised repeated CV (macro-F1={supervised_mean:.2f})",
    )
    display(Image(filename=str(supervised_path)))

    comparison_path = OUTPUT_DIR / "comparison_bars.png"
    render_score_bars(
        ["zero-shot\n(descriptive)", "supervised\n(repeated CV)"],
        [zero_shot_f1, supervised_mean],
        comparison_path,
        errors=[0.0, supervised_std],
        colors=["#bdbdbd", "#2a9d8f"],
        title="labelled clip anomaly classification",
        ylabel="macro-F1",
    )
    display(Image(filename=str(comparison_path)))
else:
    print("Supervised plots were not created because validation was skipped.")


## 7. Fit the final supervised classifier

Fit the pipeline on every labelled clip for subsequent experiments. The table below is only a
training-set sanity check. The saved bundle contains the complete PCA/classifier pipeline, class
names, orientation cutoff, frame-selection count, and exact training clip list.


In [ ]:
final_classifier = make_supervised_classifier(len(CLIP_FEATURES))
final_classifier.fit(CLIP_FEATURES, labels)
training_predictions = final_classifier.predict(CLIP_FEATURES)

# Display training predictions to catch obvious data or label problems.
training_table = clip_table[["class", "video", "rotated_180"]].copy()
training_table["training_prediction"] = [
    CLASS_NAMES[int(code)]
    for code in training_predictions
]
training_table["correct_on_training"] = training_predictions == labels
display(training_table)

model_bundle = {
    "classifier": final_classifier,
    "class_names": CLASS_NAMES,
    "n_select": N_SELECT,
    "rotate_before_timestamp": ROTATE_BEFORE_TIMESTAMP.isoformat(),
    "training_clips": [str(record["video_path"]) for record in clip_records],
    "feature_method": "mean of frozen ResNet50 embeddings over NCC-selected onset/peak frames",
}
joblib.dump(model_bundle, MODEL_PATH)
print(f"Saved supervised classifier: {MODEL_PATH}")


## 8. Classify a new clip

This helper applies the identical filename-date orientation rule, then follows decode → NCC
selection → frozen ResNet50 → mean-pool → supervised pipeline. It returns both the selected frame
indices and whether the input clip was rotated for inspection.


In [ ]:
def extract_clip_feature(video_path):
    video_path = Path(video_path)
    video_timestamp = parse_video_timestamp(video_path)
    rotate_180 = video_timestamp < ROTATE_BEFORE_TIMESTAMP

    # Match the orientation and NCC selection used to build the training features.
    frames, fps = decode_clip_rgb_with_fps(video_path)
    frames = apply_video_orientation(frames, rotate_180)
    small_frames = [cv2.resize(frame, (_NCC_DOWN, _NCC_DOWN)) for frame in frames]
    selection = select_peak_pair_window(small_frames, fps, N_SELECT)
    selected_frames = [frames[index] for index in selection.window]
    if not selected_frames:
        raise ValueError(f"NCC selected no frames for {video_path}")

    # Apply the same mean pooling and normalisation used for training clips.
    frame_features = get_resnet_embedder()(selected_frames)
    clip_feature = frame_features.mean(axis=0)
    feature_norm = np.linalg.norm(clip_feature)
    if feature_norm > 0:
        clip_feature = clip_feature / feature_norm
    return clip_feature.astype(np.float32), selection.window, rotate_180


def classify_new_clip(video_path):
    # Extract one clip feature with the same steps used during training.
    clip_feature, selected_indices, rotate_180 = extract_clip_feature(video_path)
    feature_row = clip_feature[None, :]

    # Return the predicted label and all class probabilities for reporting.
    predicted_code = int(final_classifier.predict(feature_row)[0])
    probability_values = final_classifier.predict_proba(feature_row)[0]
    class_probabilities = {
        CLASS_NAMES[int(class_code)]: float(probability)
        for class_code, probability in zip(final_classifier.classes_, probability_values)
    }
    return {
        "video_path": Path(video_path),
        "predicted_code": predicted_code,
        "predicted_label": CLASS_NAMES[predicted_code],
        "confidence": class_probabilities[CLASS_NAMES[predicted_code]],
        "probabilities": class_probabilities,
        "rotated_180": rotate_180,
        "selected_indices": selected_indices,
    }


# Example after adding an unlabelled clip:
# classify_new_clip(CNN_CLASSIFIER_DIR / "data" / "training_clips" / "normal" / "cortexvpu-01a-005-41884872_2026-07-01_06-42-50_650000_before_stop.mp4")

## 9. Classify videos from S3

This resumable batch follows the same PCA + logistic-regression inference path as Chapter 8. It
downloads one video at a time and appends every result immediately to a classifier CSV. Only anomaly
clips are saved under their predicted class folder; normal clips and temporary downloads are deleted.

Review the settings first. `S3_RUN_BATCH` is deliberately `False`, so **Run All does not contact S3**.

In [ ]:
import hashlib
import shutil

import boto3

# Keep this False until the bucket, prefix, and time range have been reviewed.
S3_RUN_BATCH = True
S3_AWS_PROFILE = "DashcamGlbDiageoProdDataContrib-522196013725"
S3_BUCKET = "diageo-prod-global-dashcam-mc-nuc-video"
S3_PREFIX = "cortexvpu-01a-005-41884872/"
S3_VIDEO_EXTENSIONS = {".ts", ".mp4", ".avi", ".mov", ".mkv", ".m4v"}
S3_START_TIME = "2026-07-01 07:00:00"
S3_END_TIME = "2026-07-02 23:15:00"
S3_TEST_VIDEO_FILENAME = None
S3_MAX_VIDEOS_TO_PROCESS = 10000
S3_REPROCESS_SUCCESSFUL_VIDEOS = False
S3_CLASSES_TO_SAVE = {"fallen_on_entry", "fell_during_view"}

# Keep S3 outputs separate from the labelled training data.
S3_CLASSIFIER_OUTPUT_DIR = OUTPUT_DIR / "s3_classifier"
S3_DOWNLOAD_DIR = S3_CLASSIFIER_OUTPUT_DIR / "downloads"
S3_RESULTS_CSV_PATH = S3_CLASSIFIER_OUTPUT_DIR / "classification_results.csv"
S3_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
for class_name in S3_CLASSES_TO_SAVE:
    (S3_CLASSIFIER_OUTPUT_DIR / class_name).mkdir(parents=True, exist_ok=True)


def append_classifier_row(result_row, csv_path):
    # Append one row directly so long batches do not repeatedly rewrite the whole CSV.
    csv_exists = csv_path.exists() and csv_path.stat().st_size > 0
    result_columns = list(result_row)

    # A pre-existing CSV must keep the same column order for safe row appends.
    if csv_exists:
        existing_columns = list(pd.read_csv(csv_path, nrows=0).columns)
        if existing_columns != result_columns:
            raise ValueError(
                "Existing classifier CSV columns do not match the current result schema. "
                f"Expected {result_columns}, found {existing_columns}."
            )

    pd.DataFrame([result_row], columns=result_columns).to_csv(
        csv_path,
        mode="a",
        header=not csv_exists,
        index=False,
    )


def load_successful_s3_keys(csv_path):
    # A successful key is skipped on future runs; failed keys remain eligible for retry.
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        return set()

    existing_result_df = pd.read_csv(csv_path)
    if "s3_key" not in existing_result_df.columns:
        return set()
    if "status" in existing_result_df.columns:
        existing_result_df = existing_result_df[
            existing_result_df["status"].astype(str).str.lower().eq("ok")
        ]
    return set(existing_result_df["s3_key"].dropna().astype(str))


def classified_clip_path(predicted_class, filename, s3_key):
    # Include a short key hash so matching filenames from different prefixes cannot overwrite.
    source_path = Path(filename)
    key_hash = hashlib.sha1(s3_key.encode("utf-8")).hexdigest()[:10]
    saved_filename = f"{source_path.stem}__{key_hash}{source_path.suffix.lower()}"
    return S3_CLASSIFIER_OUTPUT_DIR / predicted_class / saved_filename


print(f"S3 bucket:       {S3_BUCKET}")
print(f"S3 prefix:       {S3_PREFIX}")
print(f"S3 period:       {S3_START_TIME} to {S3_END_TIME}")
print(f"Results CSV:     {S3_RESULTS_CSV_PATH}")
print(f"Class folders:   {S3_CLASSIFIER_OUTPUT_DIR}")
print(f"Run enabled:     {S3_RUN_BATCH}")

In [ ]:
if not S3_RUN_BATCH:
    print("S3 batch is disabled. Review Chapter 9 settings, set S3_RUN_BATCH = True, and rerun this cell.")
    s3_video_queue_df = pd.DataFrame()
    s3_batch_results_df = pd.DataFrame()
else:
    # Connect with the configured AWS profile and list matching video metadata.
    if S3_AWS_PROFILE:
        s3_session = boto3.Session(profile_name=S3_AWS_PROFILE)
    else:
        s3_session = boto3.Session()
    s3_client = s3_session.client("s3")

    start_timestamp = pd.Timestamp(S3_START_TIME) if S3_START_TIME else None
    end_timestamp = pd.Timestamp(S3_END_TIME) if S3_END_TIME else None
    candidate_rows = []
    paginator = s3_client.get_paginator("list_objects_v2")

    # Build a lightweight queue from object metadata without downloading videos yet.
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
        for object_record in page.get("Contents", []):
            s3_key = str(object_record["Key"])
            filename = Path(s3_key).name
            if Path(filename).suffix.lower() not in S3_VIDEO_EXTENSIONS:
                continue
            if S3_TEST_VIDEO_FILENAME is not None and filename != S3_TEST_VIDEO_FILENAME:
                continue

            timestamp_match = VIDEO_TIMESTAMP_PATTERN.search(filename)
            if timestamp_match is None:
                continue
            video_timestamp = pd.to_datetime(
                timestamp_match.group(1),
                format="%Y-%m-%d_%H-%M-%S_%f",
            )
            if start_timestamp is not None and video_timestamp < start_timestamp:
                continue
            if end_timestamp is not None and video_timestamp > end_timestamp:
                continue

            candidate_rows.append(
                {
                    "s3_key": s3_key,
                    "filename": filename,
                    "video_timestamp": video_timestamp,
                    "size_bytes": int(object_record.get("Size", 0)),
                }
            )

    s3_video_queue_df = pd.DataFrame(
        candidate_rows,
        columns=["s3_key", "filename", "video_timestamp", "size_bytes"],
    )
    successful_s3_keys = load_successful_s3_keys(S3_RESULTS_CSV_PATH)

    # Sort chronologically, skip completed work, then enforce the requested batch limit.
    if not s3_video_queue_df.empty:
        s3_video_queue_df = s3_video_queue_df.sort_values(
            ["video_timestamp", "filename"]
        ).reset_index(drop=True)
        if not S3_REPROCESS_SUCCESSFUL_VIDEOS:
            s3_video_queue_df = s3_video_queue_df[
                ~s3_video_queue_df["s3_key"].isin(successful_s3_keys)
            ].copy()
        if S3_MAX_VIDEOS_TO_PROCESS is not None:
            s3_video_queue_df = s3_video_queue_df.head(
                S3_MAX_VIDEOS_TO_PROCESS
            ).reset_index(drop=True)

    print(f"Previously classified successfully: {len(successful_s3_keys)}")
    print(f"Videos queued:                    {len(s3_video_queue_df)}")
    display(s3_video_queue_df.head(20))

    current_batch_rows = []
    valid_class_names = set(CLASS_NAMES.values())

    # Download, classify, record, and clean up one video before starting the next.
    for queue_index, video_row in s3_video_queue_df.iterrows():
        s3_key = str(video_row["s3_key"])
        filename = str(video_row["filename"])
        video_timestamp = pd.Timestamp(video_row["video_timestamp"])
        key_hash = hashlib.sha1(s3_key.encode("utf-8")).hexdigest()[:10]
        source_path = Path(filename)
        download_filename = f"{source_path.stem}__{key_hash}{source_path.suffix.lower()}"
        local_video_path = S3_DOWNLOAD_DIR / download_filename
        partial_download_path = local_video_path.with_suffix(local_video_path.suffix + ".part")

        base_result_row = {
            "processed_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
            "classifier": "frozen_resnet50_pca_logistic_regression",
            "model_path": str(MODEL_PATH),
            "s3_bucket": S3_BUCKET,
            "s3_key": s3_key,
            "video_name": filename,
            "video_timestamp": video_timestamp.isoformat(),
            "source_size_bytes": int(video_row["size_bytes"]),
            "n_select": N_SELECT,
        }

        try:
            print(f"[{queue_index + 1}/{len(s3_video_queue_df)}] Downloading {filename}")
            s3_client.download_file(S3_BUCKET, s3_key, str(partial_download_path))
            partial_download_path.replace(local_video_path)

            # Chapter 8 applies orientation, frame selection, ResNet, PCA, and logistic regression.
            prediction_result = classify_new_clip(local_video_path)
            predicted_class = str(prediction_result["predicted_label"])
            if predicted_class not in valid_class_names:
                raise ValueError(f"Unexpected predicted class: {predicted_class}")

            # Keep anomaly evidence, but do not retain normal videos locally.
            saved_clip_path = None
            if predicted_class in S3_CLASSES_TO_SAVE:
                saved_clip_path = classified_clip_path(predicted_class, filename, s3_key)
                shutil.copy2(local_video_path, saved_clip_path)
            class_probabilities = prediction_result["probabilities"]

            result_row = {
                **base_result_row,
                "status": "ok",
                "error": "",
                "predicted_label": predicted_class,
                "confidence": float(prediction_result["confidence"]),
                "selected_frame_count": len(prediction_result["selected_indices"]),
                "rotate_180": bool(prediction_result["rotated_180"]),
                "prob_normal": float(class_probabilities["normal"]),
                "prob_fallen_on_entry": float(class_probabilities["fallen_on_entry"]),
                "prob_fell_during_view": float(class_probabilities["fell_during_view"]),
                "saved_clip_path": str(saved_clip_path) if saved_clip_path is not None else "",
            }
            append_classifier_row(result_row, S3_RESULTS_CSV_PATH)
            current_batch_rows.append(result_row)
            if saved_clip_path is None:
                print(f"  {predicted_class} ({result_row['confidence']:.1%}) -> not saved locally")
            else:
                print(f"  {predicted_class} ({result_row['confidence']:.1%}) -> {saved_clip_path}")
        except Exception as error:
            failed_result_row = {
                **base_result_row,
                "status": "failed",
                "error": str(error),
                "predicted_label": "",
                "confidence": np.nan,
                "selected_frame_count": 0,
                "rotate_180": np.nan,
                "prob_normal": np.nan,
                "prob_fallen_on_entry": np.nan,
                "prob_fell_during_view": np.nan,
                "saved_clip_path": "",
            }
            append_classifier_row(failed_result_row, S3_RESULTS_CSV_PATH)
            current_batch_rows.append(failed_result_row)
            print(f"  Failed {filename}: {error}")
        finally:
            # Remove both complete and partial downloads to keep disk use bounded.
            if partial_download_path.exists():
                partial_download_path.unlink()
            if local_video_path.exists():
                local_video_path.unlink()

    s3_batch_results_df = pd.DataFrame(current_batch_rows)
    display(s3_batch_results_df)
    print(f"Classifier CSV:  {S3_RESULTS_CSV_PATH}")
    print(f"Classified clips: {S3_CLASSIFIER_OUTPUT_DIR}")

---

**What changed from the demo:** labelled folders replace the synthetic manifest, videos before the
configured timestamp are rotated 180°, and NCC-selected frames are mean-pooled into one frozen
ResNet50 vector per video. The final model is a conventional supervised PCA + logistic-regression
pipeline. There is no one-shot sampling.

**Current data limitation:** repeated stratified validation requires at least three independently
recorded clips in every class. More clips are strongly recommended; the threshold only prevents an
invalid evaluation from being reported while the labelled dataset is being assembled.
